# The download cache

Every feed `fetch` downloads is kept in the download cache, the `gtfs`
folder under `cache_dir` (by default the platform's cache directory), one
version per distinct archive, named by its SHA-256. A later call uses a
cached version that serves its study day instead of downloading the feed,
offline too, and a repeated request uses the version it used before, so a
repeated run delivers the same feeds and reports. A day no cached version
serves downloads the feed and keeps the older versions. The crops, repairs
and validation results made from a version are kept with it and read back
by a call that makes the same, as are the trips found repeating those of the
other feeds the call delivers.

## Reuse a download

Let's fetch the feed recommended for Turku, which the
[Quickstart](quickstart.ipynb) already downloaded:

In [1]:
import transitio

rec = transitio.place("Turku").recommend("2026-10-14")
result = transitio.fetch(feeds=rec, osm=False)
result.selection_table()[["feed_id", "cache"]]

Fetching 1 feed for Turku


[1/1] f-u6x-turunlinja~autoilijainosakeyhtiö~savonlinjaoy~sl~autolinja: cached copy reused


Done in 2 s: 1 feed delivered, 0 skipped


,feed_id,cache
0,f-u6x-turunlinja~autoilijainosakeyhtiö~savonli...,reused


Notice that the `cache` column says `reused`: the call delivered the cached
version instead of downloading the feed again.

## Download again

```python
transitio.fetch(feeds=rec, use_cache=False)
```

`use_cache=False` downloads every feed again and replaces its cached
versions; when every attempt for a feed fails, the version a cached call
would use is delivered with a warning. A warm cache can decide differently
from a cold one: a cached version that serves the day is used without the
probe `expired="skip"` sends, and a cached feed counts as unchanged since
indexed for `contained="drop"` only when that was proven under the index
snapshot in use. For a feed that needs an account, credentials for its
provider count alike whichever key they hold, and a call without them uses
only copies fetched without them.

## Keep the delivered feeds in a folder

With `directory=` the delivered feeds are copied there, each named by its
feed id: `f-nvbw~ding.zip` beside its provenance sidecar
`f-nvbw~ding.provenance.json`. An id that is not a safe file name, such as
`f-u2f-pražskáintegrovanádoprava`, becomes its ASCII form, `+` and the id's
SHA-256: `f-u2f-prazskaintegrovanadoprava+<sha256>.zip`. A later call that
delivers the same feed into the same directory replaces its files, and one
that leaves the feed out for repeating other feeds' trips removes its file;
give calls that run at the same time, or whose results you want to keep
side by side, a directory each. Without `directory=` the delivered
feeds are the files in the cache, read-only on Linux and macOS. Either way
`result.paths` maps each delivered feed's id to its file:

In [2]:
result = transitio.fetch(feeds=rec, osm=False, directory="feeds")
for feed_id, path in result.paths.items():
    print(feed_id, path)

Fetching 1 feed for Turku


[1/1] f-u6x-turunlinja~autoilijainosakeyhtiö~savonlinjaoy~sl~autolinja: cached copy reused


Done in 1 s: 1 feed delivered, 0 skipped


f-u6x-turunlinja~autoilijainosakeyhtiö~savonlinjaoy~sl~autolinja feeds/f-u6x-turunlinjaautoilijainosakeyhtiosavonlinjaoyslautolinja+ca23bf3049e064ba45a72fc64e47bf328192c3b3765e1e2eca2356351af63b34.zip


## List and clear the cache

`transitio.cache` lists what the cache holds and clears it. `info()` gives one
row per cached version, with its sizes:

In [3]:
info = transitio.cache.info()
info[["feed_id", "archive_bytes", "outputs_bytes", "served", "retrieved_at"]]

,feed_id,archive_bytes,outputs_bytes,served,retrieved_at
0,f-ud-tallinn,5679909,4959096,1,2026-10-09T13:58:16.096231+00:00
1,mdb-864,13115567,640,0,2026-10-09T14:01:10.206210+00:00
2,f-mdb-3215,529523655,29170627,1,2026-10-08T09:12:44.955173+00:00
3,f-u6x-turunlinja~autoilijainosakeyhtiö~savonli...,13135293,28898338,2,2026-10-09T14:04:05.538328+00:00
4,f-vr~fi,5610438,1181112,1,2026-10-09T13:58:06.625410+00:00
5,f-mdb-3235,48903755,26877113,1,2026-10-09T13:58:21.998541+00:00


`clear()` removes versions, all of them or those unused for a while:

```python
import datetime

transitio.cache.clear(older_than=datetime.timedelta(days=30))  # unused 30 days
transitio.cache.clear()       # everything
```

`clear` waits for a fetch working on a feed; a feed a running fetch adds
after `clear` started is left for the next call.